In [12]:
# multivariate lstm example
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy import array
from numpy import hstack
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from keras.layers import Input
from tqdm import tqdm
import os
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
# Configure TensorFlow to use all available CPUs
tf.config.threading.set_intra_op_parallelism_threads(0)  # Use all CPU threads for operations
tf.config.threading.set_inter_op_parallelism_threads(0)  # Use all CPU threads for inter-operation parallelism

os.chdir('/tf/Capstone/')

In [ ]:
# Import the relevant data from the database
def feature_data_retrieve():
    # Make the connection to the database
    conn = sqlite3.connect('data.sqlite')
    # Write the query
    query = '''SELECT station_id, docks_available, timestamp, year, month, day, hour, temperature_2m, apparent_temperature, relative_humidity_2m, wind_speed_10m, sunshine_duration, rain FROM data_table WHERE year IN (2022, 2023, 2024) AND month IN (1, 2, 3, 4, 5) AND station_id BETWEEN 1 AND 496'''
    # Read the query into a pandas dataframe
    for chunk in pd.read_sql_query(query, conn, chunksize=10000):
        yield pd.DataFrame(chunk)
    # Close the connection
    conn.close()

feature_data = pd.DataFrame()

# Load the data
for chunk in tqdm(feature_data_retrieve()):
    feature_data = pd.concat([feature_data, chunk])

# Convert the timestamp column to datetime
feature_data['timestamp'] = pd.to_datetime(feature_data['timestamp'])

# Make timestamp the index
feature_data.set_index('timestamp', inplace=True)

# Sort the feature data by station_id and then station_id by timestamp
feature_data.sort_values(by=['station_id', 'timestamp'], inplace=True)

# Engineer features, four columns which will have docks available shifted by 1, 2, 3, and 4 hours before the current timestamp. They are called ctx-1, ctx-2, ctx-3, and ctx-4
feature_data['ctx-1'] = feature_data.groupby('station_id')['docks_available'].shift(1)
feature_data['ctx-2'] = feature_data.groupby('station_id')['docks_available'].shift(2)
feature_data['ctx-3'] = feature_data.groupby('station_id')['docks_available'].shift(3)
feature_data['ctx-4'] = feature_data.groupby('station_id')['docks_available'].shift(4)


# Drop rows with missing values
feature_data.dropna(inplace=True)

# Drop columns that are not needed
feature_data.drop(columns=['relative_humidity_2m'], inplace=True)
feature_data.drop(columns=['temperature_2m'], inplace=True)

# Convert the timestamp into a column again and reset the index
feature_data.reset_index(inplace=True)



# Normalize the columns apparent_temperature, wind_speed_10m, sunshine_duration, and rain
feature_data['apparent_temperature'] = (feature_data['apparent_temperature'] - feature_data['apparent_temperature'].min()) / (feature_data['apparent_temperature'].max() - feature_data['apparent_temperature'].min())
feature_data['wind_speed_10m'] = (feature_data['wind_speed_10m'] - feature_data['wind_speed_10m'].min()) / (feature_data['wind_speed_10m'].max() - feature_data['wind_speed_10m'].min())
feature_data['sunshine_duration'] = (feature_data['sunshine_duration'] - feature_data['sunshine_duration'].min()) / (feature_data['sunshine_duration'].max() - feature_data['sunshine_duration'].min())
feature_data['rain'] = (feature_data['rain'] - feature_data['rain'].min()) / (feature_data['rain'].max() - feature_data['rain'].min())

display(feature_data.head(10))

352it [00:50,  7.03it/s]


,timestamp,station_id,docks_available,year,month,day,hour,apparent_temperature,wind_speed_10m,sunshine_duration,rain,ctx-1,ctx-2,ctx-3,ctx-4
0,2022-01-01 04:00:00,1,0.717391,2022,1,1,4,0.251886,0.277307,0.000000,0.0,0.717391,0.737319,0.726449,0.682971
1,2022-01-01 05:00:00,1,0.672101,2022,1,1,5,0.253480,0.295135,0.000000,0.0,0.717391,0.717391,0.737319,0.726449
2,2022-01-01 06:00:00,1,0.630435,2022,1,1,6,0.256339,0.281550,0.000000,0.0,0.672101,0.717391,0.717391,0.737319
3,2022-01-01 07:00:00,1,0.617754,2022,1,1,7,0.259774,0.273376,0.000000,0.0,0.630435,0.672101,0.717391,0.717391
4,2022-01-01 08:00:00,1,0.614130,2022,1,1,8,0.258832,0.279344,0.000000,0.0,0.617754,0.630435,0.672101,0.717391
5,2022-01-01 09:00:00,1,0.583333,2022,1,1,9,0.270961,0.305433,0.080428,0.0,0.614130,0.617754,0.630435,0.672101
6,2022-01-01 10:00:00,1,0.625000,2022,1,1,10,0.351829,0.331956,1.000000,0.0,0.583333,0.614130,0.617754,0.630435
7,2022-01-01 11:00:00,1,0.670290,2022,1,1,11,0.468272,0.251170,1.000000,0.0,0.625000,0.583333,0.614130,0.617754
8,2022-01-01 12:00:00,1,0.737319,2022,1,1,12,0.576417,0.177313,1.000000,0.0,0.670290,0.625000,0.583333,0.614130
9,2022-01-01 13:00:00,1,0.751812,2022,1,1,13,0.583619,0.231523,1.000000,0.0,0.737319,0.670290,0.625000,0.583333


In [ ]:
# Use 2024 for testing and 2022-2023 for training
train_data = feature_data[feature_data['year'].isin([2022, 2023])]
test_data = feature_data[feature_data['year'].isin([2024])]

# Drop the timestamp column
train_data.drop(columns=['timestamp'], inplace=True)
test_data.drop(columns=['timestamp'], inplace=True)

# Make station_id categorical
train_data['station_id'] = train_data['station_id'].astype('category')
test_data['station_id'] = test_data['station_id'].astype('category')

FEATURES = ['docks_available', 'ctx-1', 'ctx-2', 'ctx-3', 'ctx-4', 'apparent_temperature', 'wind_speed_10m', 'sunshine_duration', 'rain']
TARGET = 'docks_available'

x = train_data[FEATURES].values
y = train_data[TARGET].values

# Normalize the features
scaler = MinMaxScaler(feature_range=(0, 1))
x = scaler.fit_transform(x)
y = scaler.fit_transform(y.reshape(-1, 1))
# Reshape the input data to be 3D
x = x.reshape((x.shape[0], 1, x.shape[1]))
# Reshape the output data to be 2D
y = y.reshape((y.shape[0], 1))




/tmp/ipykernel_782/3920577062.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data.drop(columns=['timestamp'], inplace=True)
/tmp/ipykernel_782/3920577062.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data.drop(columns=['timestamp'], inplace=True)
/tmp/ipykernel_782/3920577062.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['station_id'] = tra

[[[0.7173913  0.7173913  0.73731884 ... 0.27730721 0.         0.        ]]

 [[0.67210145 0.7173913  0.7173913  ... 0.29513456 0.         0.        ]]

 [[0.63043478 0.67210145 0.7173913  ... 0.28154961 0.         0.        ]]

 ...

 [[0.03703704 0.36574074 0.27777778 ... 0.14358423 0.         0.        ]]

 [[0.14814815 0.03703704 0.36574074 ... 0.14358423 0.         0.        ]]

 [[0.00925926 0.14814815 0.03703704 ... 0.12393241 0.         0.        ]]]
[[0.7173913 ]
 [0.67210145]
 [0.63043478]
 ...
 [0.03703704]
 [0.14814815]
 [0.00925926]]


In [ ]:
# Build the LSTM model
# Build the LSTM model
model = Sequential()
model.add(Input(shape=(x.shape[1], x.shape[2])))  # Define input shape using Input layer
model.add(LSTM(50, return_sequences=False))  # LSTM layer
model.add(Dense(1))  # Output layer
model.compile(loss='mse', optimizer='adam')

# Train the model
model.fit(x, y, epochs=50, batch_size=32, verbose=2)

# Make predictions
yhat = model.predict(x)

# Evaluate the model
train_score = model.evaluate(x, y, verbose=0)
print('Train Score: %.2f RMSE' % (train_score**0.5))

Epoch 1/50


In [ ]:
"Lgb;/.h,